In [ ]:
import sys
import os
from pathlib import Path

# Add project root to path
project_root = Path.cwd().parent.parent
sys.path.insert(0, str(project_root))

import torch
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from torch.utils.data import DataLoader
import torchvision.transforms as transforms
from sklearn.metrics import confusion_matrix, accuracy_score, classification_report

# Import project modules
from benchmarks.medMNIST.utils import train_models_load_datasets as tr
from medmnist import INFO

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA device: {torch.cuda.get_device_name(0)}")

## 1. Configuration

In [ ]:
# Configuration
flag = 'organamnist'
model_backbone = 'resnet18'
setup = ''  # Standard training (no DA/DO/DADO)
image_size = 224
batch_size = 128
device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')

print(f"Dataset: {flag}")
print(f"Model: {model_backbone}")
print(f"Setup: {setup if setup else 'standard'}")
print(f"Device: {device}")

## 2. Load Dataset

In [ ]:
# RepeatGrayToRGB transform for grayscale datasets
class RepeatGrayToRGB:
    """Transform for converting grayscale to RGB."""
    def __call__(self, x):
        return x.repeat(3, 1, 1)

# OrganAMNIST is grayscale
color = False

# Define transforms (same as in run_medmnist_benchmark.py)
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=[.5], std=[.5]),
    RepeatGrayToRGB(),
])

# Load datasets
print("\nLoading OrganAMNIST datasets...")
[study_dataset, calib_dataset, test_dataset], \
[_, calib_loader, test_loader], info = \
    tr.load_datasets(flag, color, image_size, transform, batch_size)

print(f"\nDataset sizes:")
print(f"  Training: {len(study_dataset)}")
print(f"  Calibration: {len(calib_dataset)}")
print(f"  Test: {len(test_dataset)}")
print(f"\nTask: {info['task']}")
print(f"Number of classes: {len(info['label'])}")
print(f"\nClass labels:")
for idx, label in info['label'].items():
    print(f"  {idx}: {label}")

## 3. Load Models

In [ ]:
# Load the 5 CV fold models
print("\nLoading models...")
models = tr.load_models(
    flag=flag,
    device=device,
    size=image_size,
    model_backbone=model_backbone,
    setup=setup
)

print(f"Loaded {len(models)} models (CV folds)")

# Set all models to eval mode
for model in models:
    model.eval()

## 4. Evaluate on Calibration Set

In [ ]:
# Collect predictions from all models (ensemble)
print("\nEvaluating ensemble on calibration set...")

y_true_all = []
y_pred_all = []
y_probs_all = []

with torch.no_grad():
    for batch_idx, (images, labels) in enumerate(calib_loader):
        images = images.to(device)
        labels = labels.cpu().numpy().squeeze()
        
        # Collect predictions from all 5 models
        batch_probs = []
        for model in models:
            logits = model(images)
            probs = torch.softmax(logits, dim=1)
            batch_probs.append(probs.cpu().numpy())
        
        # Average predictions across models (ensemble)
        ensemble_probs = np.mean(batch_probs, axis=0)  # (batch_size, num_classes)
        ensemble_preds = np.argmax(ensemble_probs, axis=1)
        
        y_true_all.extend(labels)
        y_pred_all.extend(ensemble_preds)
        y_probs_all.append(ensemble_probs)
        
        if (batch_idx + 1) % 20 == 0:
            print(f"  Processed {(batch_idx + 1) * batch_size} / {len(calib_dataset)} samples")

y_true = np.array(y_true_all)
y_pred = np.array(y_pred_all)
y_probs = np.vstack(y_probs_all)

print(f"\nCompleted evaluation on {len(y_true)} samples")

## 5. Compute Metrics

In [ ]:
# Compute accuracy
accuracy = accuracy_score(y_true, y_pred)
print(f"\nCalibration Set Accuracy: {accuracy:.4f}")

# Count correct and incorrect predictions
correct_idx = np.where(y_pred == y_true)[0]
incorrect_idx = np.where(y_pred != y_true)[0]

print(f"\nCorrect predictions: {len(correct_idx)} ({len(correct_idx)/len(y_true)*100:.2f}%)")
print(f"Incorrect predictions: {len(incorrect_idx)} ({len(incorrect_idx)/len(y_true)*100:.2f}%)")

## 6. Confusion Matrix Visualization

In [ ]:
# Compute confusion matrix
cm = confusion_matrix(y_true, y_pred)
class_names = list(info['label'].values())

# Plot confusion matrix
plt.figure(figsize=(12, 10))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=class_names, yticklabels=class_names,
            cbar_kws={'label': 'Count'})
plt.xlabel('Predicted Label', fontsize=12)
plt.ylabel('True Label', fontsize=12)
plt.title(f'Confusion Matrix - {flag.upper()} Calibration Set\n'
          f'Model: {model_backbone}, Accuracy: {accuracy:.4f}', fontsize=14)
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

print("\nConfusion Matrix:")
print(cm)

## 7. Normalized Confusion Matrix

In [ ]:
# Normalize confusion matrix (row-wise)
cm_normalized = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]

plt.figure(figsize=(12, 10))
sns.heatmap(cm_normalized, annot=True, fmt='.3f', cmap='Blues',
            xticklabels=class_names, yticklabels=class_names,
            cbar_kws={'label': 'Proportion'})
plt.xlabel('Predicted Label', fontsize=12)
plt.ylabel('True Label', fontsize=12)
plt.title(f'Normalized Confusion Matrix - {flag.upper()} Calibration Set\n'
          f'Model: {model_backbone}, Accuracy: {accuracy:.4f}', fontsize=14)
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

## 8. Classification Report

In [ ]:
# Print detailed classification report
print("\nClassification Report:")
print("=" * 80)
report = classification_report(y_true, y_pred, target_names=class_names, digits=4)
print(report)

## 9. Per-Class Error Analysis

In [ ]:
# Analyze errors per class
print("\nPer-Class Analysis:")
print("=" * 80)
print(f"{'Class':<20} {'Total':<8} {'Correct':<8} {'Incorrect':<8} {'Accuracy':<10}")
print("-" * 80)

for class_idx, class_name in enumerate(class_names):
    class_mask = (y_true == class_idx)
    class_total = class_mask.sum()
    class_correct = ((y_true == class_idx) & (y_pred == class_idx)).sum()
    class_incorrect = class_total - class_correct
    class_accuracy = class_correct / class_total if class_total > 0 else 0
    
    print(f"{class_name:<20} {class_total:<8} {class_correct:<8} {class_incorrect:<8} {class_accuracy:<10.4f}")

print("=" * 80)

## 10. Most Common Misclassifications

In [ ]:
# Find most common misclassifications
print("\nTop 10 Most Common Misclassifications:")
print("=" * 80)
print(f"{'True Class':<20} {'Predicted Class':<20} {'Count':<8}")
print("-" * 80)

# Get all misclassification pairs
misclass_pairs = []
for i in range(len(class_names)):
    for j in range(len(class_names)):
        if i != j and cm[i, j] > 0:
            misclass_pairs.append((i, j, cm[i, j]))

# Sort by count
misclass_pairs.sort(key=lambda x: x[2], reverse=True)

# Display top 10
for true_idx, pred_idx, count in misclass_pairs[:10]:
    true_name = class_names[true_idx]
    pred_name = class_names[pred_idx]
    print(f"{true_name:<20} {pred_name:<20} {int(count):<8}")

print("=" * 80)

## 11. Save Results

In [ ]:
# Save confusion matrix figure
output_dir = Path.cwd() / 'analysis_outputs'
output_dir.mkdir(exist_ok=True)

fig, ax = plt.subplots(figsize=(12, 10))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=class_names, yticklabels=class_names,
            ax=ax, cbar_kws={'label': 'Count'})
ax.set_xlabel('Predicted Label', fontsize=12)
ax.set_ylabel('True Label', fontsize=12)
ax.set_title(f'Confusion Matrix - {flag.upper()} Calibration Set\n'
             f'Model: {model_backbone}, Accuracy: {accuracy:.4f}', fontsize=14)
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()

save_path = output_dir / f'{flag}_{model_backbone}_calibration_confusion_matrix.png'
plt.savefig(save_path, dpi=300, bbox_inches='tight')
print(f"\nConfusion matrix saved to: {save_path}")
plt.show()

# Save numerical results
results = {
    'confusion_matrix': cm,
    'y_true': y_true,
    'y_pred': y_pred,
    'y_probs': y_probs,
    'accuracy': accuracy,
    'class_names': class_names
}

results_path = output_dir / f'{flag}_{model_backbone}_calibration_results.npz'
np.savez(results_path, **results)
print(f"Results saved to: {results_path}")